# SWAP-Stress: source data — units, conversions, coverage

An onboarding tour of the **observation sources** behind SWAP-Stress: GSHP, NCSS,
MT Mesonet, ReESH, and LaCADIAN.

What this notebook shows:

1. What the source registry knows about each source, and where its files land
2. The unit conversions that harmonize every source onto `suction_cm`, `theta`, `depth_cm`
3. The physical filters that decide what gets dropped
4. Stage 00 (`swapstress-standardize`) resolved as a dry run
5. Distributions of the standardized observations
6. Where the sites are

Every code cell calls `swapstress`; nothing here reimplements pipeline logic. The
modules on show are `swapstress.sources.registry`, `swapstress.sources.standardize`,
`swapstress.sources.ncss`, `swapstress.units`, and
`swapstress.features.build_training_table`.

In [ ]:
from __future__ import annotations

import os
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from swapstress.sources.registry import (
    DEFAULT_SOURCES,
    SOURCES,
    DataPaths,
    get_source,
    list_sources,
)

# ---------------------------------------------------------------------------
# The one path you may have to change: where the project data tree is mounted.
# Everything below resolves from it through swapstress.sources.registry, so no
# other cell in this notebook contains a literal path.
# ---------------------------------------------------------------------------
DATA_ROOT = os.environ.get("SWAPSTRESS_DATA_ROOT", "/nas/soils")

# The released scale. "9km_conus" and the historical "250m" also resolve.
SCALE = "9km_global"

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)


def paths_for(name: str) -> DataPaths:
    """Resolve one source's on-disk layout under DATA_ROOT."""
    return DataPaths(DATA_ROOT, get_source(name), scale=SCALE)


print("DATA_ROOT:", DATA_ROOT)
print("SCALE:    ", SCALE)
print("sources the released model trains on:", ", ".join(DEFAULT_SOURCES))

## 1) The source registry

`swapstress/sources/registry.py` is the one place that knows a source's layout —
its raw inputs, its site shapefile, its MGRS index, its Earth Engine export
prefix, and where its standardized observations and feature table land. Adding a
source is a registry entry plus one standardizer; no pipeline stage hardcodes a
path.

`rosetta` is registered alongside the five observation sources, but it is a
gridded pedotransfer prior used for pretraining rather than an observation
source, so it is opted into explicitly rather than picked up by default.

In [ ]:
list_sources()

In [ ]:
rows = []
for name in SOURCES:
    src = get_source(name)
    p = paths_for(name)
    rows.append(
        {
            "source": name,
            "index_col": src.index_col,
            "has_vg_params": src.has_vg_params,
            "vg_source": src.vg_source,
            "in_default_set": name in DEFAULT_SOURCES,
            "raw_dir": p.raw_dir,
            "standardized_dir": p.preprocessed_dir,
            "curve_fits_dir": p.fit_results_dir,
            "features_table": p.ee_table,
            "sites_shapefile": p.shapefile,
        }
    )

pd.DataFrame(rows).set_index("source")

## 2) Unit conversions and the standard schema

Every source lands on the same three columns:

- `suction_cm` — suction head in **cm H₂O**, positive and rising as soil dries
- `theta` — volumetric water content as a **0–1 fraction**
- `depth_cm` — depth in **cm**

Each source arrives in its own units. The conversions live in
`swapstress.sources.standardize` (one `standardize_<source>` function per source)
and `swapstress.sources.ncss`; the constants below are read from those modules
rather than restated here.

In [ ]:
from swapstress import units
from swapstress.sources import ncss, standardize

KPA_TO_CM = standardize.MPA_TO_CM / 1000.0  # what the kPa sources multiply by

unit_table = pd.DataFrame(
    [
        {
            "source": "GSHP",
            "raw_units": "m H2O",
            "to_suction_cm": "abs(lab_head_m) * 100",
            "code": "swapstress.sources.standardize.standardize_gshp",
        },
        {
            "source": "NCSS",
            "raw_units": "bar",
            "to_suction_cm": f"bar * {ncss.BAR_TO_CM}",
            "code": "swapstress.sources.ncss.ncss_to_standardized",
        },
        {
            "source": "MT Mesonet",
            "raw_units": "kPa",
            "to_suction_cm": f"abs(KPA) * {KPA_TO_CM:.5f}",
            "code": "swapstress.sources.standardize.standardize_mt_mesonet",
        },
        {
            "source": "ReESH",
            "raw_units": "MPa",
            "to_suction_cm": f"abs(MPa_Abs) * {standardize.MPA_TO_CM}",
            "code": "swapstress.sources.standardize.standardize_reesh",
        },
        {
            "source": "LaCADIAN",
            "raw_units": "kPa",
            "to_suction_cm": f"abs(KPA) * {KPA_TO_CM:.5f}",
            "code": "swapstress.sources.standardize.standardize_lacadian",
        },
    ]
)
unit_table

### 2a) The MPa shift

The model's target is `log10(suction_cm)`, and the released product also carries
a signed `matric_potential_MPa` band. The two are the *same number* under a
change of unit: because the target is a base-10 logarithm, converting to MPa is
an exact additive shift in log space. R², RMSE, and interval widths in log units
are identical either way — which is why stage 07 does the conversion as a
postprocess and nothing retrains.

`swapstress/units.py` is the only place the constant lives.

In [ ]:
print(f"units.MPA_TO_CM       = {units.MPA_TO_CM}")
print(f"units.LOG10_MPA_TO_CM = {units.LOG10_MPA_TO_CM:.7f}   (= log10(MPA_TO_CM))")
print()

for log10_cm in [1.0, 2.0, 3.0, 4.1832, 5.0]:
    psi_mpa = units.log10_suction_cm_to_mpa(log10_cm)
    log10_abs = units.log10_suction_cm_to_log10_abs_mpa(log10_cm)
    print(
        f"log10_suction_cm={log10_cm:6.4f}  ->  "
        f"{psi_mpa:12.6f} MPa   log10|MPa|={log10_abs:8.4f}"
    )

print("\nPermanent wilting point, -1.5 MPa, as suction head:")
print(f"  {units.mpa_to_suction_cm(-1.5):,.0f} cm H2O")

## 3) Physical sanity filters

Standardization applies physical bounds and drops what falls outside them.
`swapstress.sources.standardize.apply_physical_filters` is the shared
implementation; every `standardize_<source>` function calls it and prints what it
dropped. The constants below are the authoritative thresholds.

`KPA_MAX` applies only to the in-situ sources (MT Mesonet, LaCADIAN), where it is
the field sensor's usable range rather than a physical limit.

In [ ]:
filters = pd.DataFrame(
    [
        {
            "module": "swapstress.sources.standardize",
            "SUCTION_CM_MAX": standardize.SUCTION_CM_MAX,
            "THETA_MIN": standardize.THETA_MIN,
            "THETA_MAX": standardize.THETA_MAX,
            "KPA_MAX": standardize.KPA_MAX,
            "BULK_DENSITY_MIN": standardize.BULK_DENSITY_MIN,
            "BULK_DENSITY_MAX": standardize.BULK_DENSITY_MAX,
        },
        {
            "module": "swapstress.sources.ncss",
            "SUCTION_CM_MAX": ncss.SUCTION_CM_MAX,
            "THETA_MIN": ncss.THETA_MIN,
            "THETA_MAX": ncss.THETA_MAX,
            "KPA_MAX": None,
            "BULK_DENSITY_MIN": ncss.BULK_DENSITY_MIN,
            "BULK_DENSITY_MAX": ncss.BULK_DENSITY_MAX,
        },
    ]
).set_index("module")
filters

## 4) Stage 00 — `swapstress-standardize`

Stage 00 reads each source's raw files and writes one standardized CSV per
profile or station to `<data_root>/soil_potential_obs/preprocessed/<source>/`.
The real invocation is:

```bash
uv run swapstress-standardize --data-root /nas/soils
```

The dry run below resolves every raw input and standardized output for every
source and marks which inputs are present. It reads nothing and writes nothing,
so it is safe to run anywhere.

In [ ]:
standardize.main(["--data-root", DATA_ROOT, "--dry-run"])

In [ ]:
for name in DEFAULT_SOURCES:
    d = paths_for(name).preprocessed_dir
    files = sorted(glob(os.path.join(d, "*.csv"))) if os.path.isdir(d) else []
    print(f"{name:<12s} {len(files):>7,d} standardized CSVs   {d}")

## 5) Distributions of the standardized observations

`swapstress.features.build_training_table.load_observations_for_source` is the
loader stage 02 uses: it reads a source's standardized CSVs (falling back to the
fitted curve JSONs for any profile without one) and returns one row per
(theta, suction) observation.

This is the pre-feature view — observations only, no covariates joined yet.

In [ ]:
from swapstress.features.build_training_table import load_observations_for_source

frames = []
for name in DEFAULT_SOURCES:
    obs = load_observations_for_source(get_source(name), DATA_ROOT)
    print(f"{name:<12s} {len(obs):>9,d} observations")
    if obs.empty:
        continue
    obs = obs.assign(source=name)
    frames.append(obs)

obs_all = pd.concat(frames, ignore_index=True)
print(f"\ntotal {len(obs_all):,} observations, columns: {list(obs_all.columns)}")
obs_all.head()

In [ ]:
plot_df = obs_all.assign(log10_suction_cm=np.log10(obs_all["suction_cm"]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=120)
for ax, col, label in zip(
    axes,
    ["theta", "log10_suction_cm", "depth_cm"],
    ["theta (0-1)", "log10(suction_cm)", "depth (cm)"],
):
    for name, g in plot_df.groupby("source"):
        vals = g[col].dropna().to_numpy()
        if vals.size:
            ax.hist(vals, bins=60, alpha=0.4, density=True, label=name)
    ax.set_xlabel(label)
    ax.legend(fontsize=8)
axes[0].set_ylabel("density")

fig.tight_layout()
out = os.path.join(OUT_DIR, "source_data_histograms.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 6) Where the sites are

Each source's site geometry comes from the registry
(`DataPaths.shapefile`), so this map follows whatever the registry currently
points at. State outlines come from `swapstress.figures.basemap`, which resolves
the boundaries tree the same way the source registry resolves the MGRS index.

In [ ]:
import geopandas as gpd

from swapstress.figures import basemap

# Not part of the packaged basemap assets, which are CONUS-only. Point this at a
# world outline of your choosing, or set it to None to draw the global panel bare.
WORLD_SHP = os.path.join(
    basemap.boundaries_root(),
    "boundaries",
    "world_countries",
    "World_Countries_shp.shp",
)


def conus_axis(ax):
    """Frame a lon/lat axis on CONUS with state outlines. Display plumbing.

    The descriptor's map figures project to Albers and style their own axes;
    this is the quick notebook equivalent, so it corrects the aspect by
    latitude rather than projecting.
    """
    basemap.load_conus_states().boundary.plot(ax=ax, color="0.5", linewidth=0.4)
    ax.set_xlim(*basemap.CONUS_LON)
    ax.set_ylim(*basemap.CONUS_LAT)
    ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(basemap.CONUS_LAT))))
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


MARKERS = {
    "gshp": ("s", "#2ca02c"),
    "ncss": ("d", "#9467bd"),
    "mt_mesonet": ("o", "#1f77b4"),
    "reesh": ("^", "#d62728"),
    "lacadian": ("v", "#ff7f0e"),
}

layers = []
for name in DEFAULT_SOURCES:
    shp = paths_for(name).shapefile
    if not (shp and os.path.exists(shp)):
        print(f"{name}: no site shapefile at {shp}")
        continue
    gdf = gpd.read_file(shp)
    gdf = gdf.to_crs(4326) if gdf.crs else gdf.set_crs(4326)
    layers.append((name, gdf))
    print(f"{name:<12s} {len(gdf):>6,d} sites   {shp}")

In [ ]:
fig, (ax_g, ax_c) = plt.subplots(2, 1, figsize=(11, 9), dpi=130)

if WORLD_SHP and os.path.exists(WORLD_SHP):
    gpd.read_file(WORLD_SHP).to_crs(4326).boundary.plot(
        ax=ax_g, color="0.6", linewidth=0.3
    )
ax_g.set_title("Global")
ax_g.set_xlim(-180, 180)
ax_g.set_ylim(-60, 85)
ax_g.set_xlabel("Longitude")
ax_g.set_ylabel("Latitude")

conus_axis(ax_c)
ax_c.set_title("CONUS")

for name, gdf in layers:
    marker, color = MARKERS[name]
    xs, ys = gdf.geometry.x, gdf.geometry.y
    ax_g.scatter(
        xs,
        ys,
        s=5,
        marker=marker,
        color=color,
        alpha=0.6,
        label=f"{name} (n={len(gdf)})",
    )
    ax_c.scatter(xs, ys, s=8, marker=marker, color=color, alpha=0.7, label=name)

ax_g.legend(loc="lower left", fontsize=8)
ax_c.legend(loc="lower left", fontsize=8)

fig.tight_layout(h_pad=1.5)
out = os.path.join(OUT_DIR, "source_site_maps.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## Next

`02_earth_engine.ipynb` builds the covariate stack these sites get sampled
against (stage 01), and `04_training_table.ipynb` joins the two into the
observation-level training table (stage 02).